# Panel Getting Started

This notebook is adapted from official Panel getting-started examples:
- https://github.com/holoviz/panel/blob/main/scripts/jupyterlite/files/Getting_Started.ipynb
- https://github.com/holoviz/panel/blob/main/doc/how_to/components/construct_panes.md

In [1]:
import sys, types, warnings

# Suppress Bokeh model re-registration warnings that appear when this cell
# is re-run (panel modules are cleared below but bokeh stays loaded, so the
# next panel import triggers "Duplicate qualified model definition" for every
# bokeh model class that panel re-imports).
warnings.filterwarnings('ignore', message='.*Duplicate.*model.*', category=UserWarning)

# ── Remove stale state from any previous failed attempts ──────────────
sys.modules.pop('_pyodide', None)
for _k in list(sys.modules):
    if _k == 'panel' or _k.startswith('panel.'):
        del sys.modules[_k]

class _Stub:
    def __getattr__(self, _): return _Stub()
    def __call__(self, *a, **kw): return _Stub()
    def __iter__(self): return iter([])
    def __bool__(self): return False

sys.modules['_pyodide'] = types.ModuleType('_pyodide')

_pyo = sys.modules.get('pyodide')
if _pyo is not None:
    _code = types.ModuleType('pyodide.code')
    _code.run_js = lambda src: _Stub()
    _pyo.code = _code
    sys.modules['pyodide.code'] = _code

_ffi = sys.modules.get('pyodide.ffi')
if _ffi is not None:
    if not hasattr(_ffi, 'create_proxy'):
        _ffi.create_proxy = lambda fn: fn
    if not hasattr(_ffi, 'jsnull'):
        _ffi.jsnull = None

import panel as pn
from bokeh.plotting import figure

# ── xeus-python comm compatibility ────────────────────────────────────
import pyviz_comms as _pvc

def _xeus_server_comm_init(self):
    from comm import create_comm
    self._comm = create_comm(target_name=self.id, data={})
    self._comm.on_msg(self._handle_msg)
    if self._on_open:
        self._on_open({})

def _xeus_client_comm_init(self, id=None, on_msg=None, on_error=None,
                            on_stdout=None, on_open=None):
    from comm import get_comm_manager
    _pvc.Comm.__init__(self, id, on_msg, on_error, on_stdout, on_open)
    self.manager = get_comm_manager()
    self.manager.register_target(self.id, self._handle_open)

if not getattr(_pvc.JupyterComm, '_xeus_patched', False):
    _pvc.JupyterComm.init = _xeus_server_comm_init
    _pvc.JupyterComm._xeus_patched = True

if not getattr(_pvc.JupyterCommJS, '_xeus_patched', False):
    _pvc.JupyterCommJS.__init__ = _xeus_client_comm_init
    _pvc.JupyterCommJS._xeus_patched = True

import panel.io.pyodide as _panel_pyodide
_panel_pyodide._IN_WORKER = True

pn.extension()
print(f"Panel {pn.__version__} loaded  |  _is_pyodide={pn.state._is_pyodide}")

Panel 1.8.10 loaded  |  _is_pyodide=True


In [2]:
# Official-style widget + bind pattern
w = pn.widgets.FloatSlider(name="Value", start=0, end=3.14, step=0.01, value=1.57)
pn.Row(w, pn.bind(pn.pane.Str, w))

Row
    [0] FloatSlider(end=3.14, name='Value', step=0.01, value=1.57)
    [1] ParamFunction(function, _pane=Str, defer_load=False)

In [3]:
# Explicit pane construction example
pn.pane.Markdown("""
## Quick Pane Demo
Move the slider and the text/plot below update reactively.
""")

Markdown(str)

In [4]:
import math
from bokeh.models import ColumnDataSource, CustomJS, Slider
from bokeh.layouts import column as bk_column

x = [i / 20 for i in range(126)]
_src = ColumnDataSource({'x': x, 'y': [math.sin(1.0 * v) for v in x]})

_p = figure(width=520, height=280, title="sin(1.0 × x)")
_p.line('x', 'y', source=_src, line_width=2, color="#0ea5e9")

_freq = Slider(title="Frequency", value=1.0, start=0.5, end=5.0, step=0.1, width=520)
_freq.js_on_change('value', CustomJS(
    args=dict(src=_src, x=x, title=_p.title),
    code="""
        const f = cb_obj.value;
        src.data.y = x.map(v => Math.sin(f * v));
        src.change.emit();
        title.text = 'sin(' + f.toFixed(1) + ' × x)';
    """
))

pn.Column(
    pn.pane.Markdown("### Reactive Bokeh Plot"),
    pn.pane.Bokeh(bk_column(_freq, _p)),
)

Column
    [0] Markdown(str)
    [1] Bokeh(Column)